# Amazon ML Challenge 2026 — Kaggle Run
CPU-first pipeline. GPU optional for cross-encoder reranker only.
Attach your dataset: Settings > Data > Add Input > your private 1GB dataset.
Expected input path: `/kaggle/input/<your-dataset>/...` containing `train/`, `test/`.


In [ ]:
# Install base + optional advanced deps
!pip install -q lightgbm rapidfuzz
# Optional: for FAISS dense retrieval + cross-encoder reranker (GPU recommended for CE)
# !pip install -q faiss-cpu transformers torch sentence-transformers
import os
DATA_DIR = '/kaggle/input/datasets/ayushastiker/amazon-ml-challenge/student_resource/dataset'  # <-- EDIT THIS
SRC = '/kaggle/working/code_ber/business_entity_resolution/src'
OUT = '/kaggle/working/output'
os.makedirs(OUT, exist_ok=True)
print(os.listdir(DATA_DIR))
print(os.listdir(os.path.join(DATA_DIR,'train'))[:10])

In [ ]:
# Phase 1 — audit (CPU, ~2-5 min). S1 2.2M, singleton 5.6%, floor F0.5 0.0558
!python $SRC/audit.py --data-dir $DATA_DIR

In [ ]:
# Phase 2 — blocking recall on sample (CPU). Tune max-df/min-len to get recall >= 0.90
!python $SRC/pipeline.py --help
!python $SRC/bench_blocking.py --data-dir $DATA_DIR --n 5000 --max-df 100 --min-len 5

In [ ]:
# Phase 3 — sampled train: matcher + calibration + threshold (CPU). No test inference.
!python $SRC/bench_train.py --data-dir $DATA_DIR --n 5000 --max-df 100 --min-len 5 --n-splits 3 --neg-per-pos-cap 10

In [ ]:
# Phase 4 — full sampled run with test inference (chunked).
# Base run (token+prefix blocking, LightGBM, isotonic, threshold tune):
!python $SRC/pipeline.py \
  --data-dir $DATA_DIR \
  --out-dir $OUT \
  --n-splits 3 \
  --sample-s1 20000 \
  --max-df 50 \
  --min-len 5 \
  --neg-per-pos-cap 10 \
  --test-chunk-size 50000 \
  --cache-dir /kaggle/working/cache \
  --save-model-dir /kaggle/working/models_20k \
  --validate

In [ ]:
# Phase 4b — same with FAISS dense retrieval (adds ~5-10% recall for typos)
# Requires: !pip install -q faiss-cpu
# !python $SRC/pipeline.py \
#   --data-dir $DATA_DIR \
#   --out-dir $OUT_faiss \
#   --n-splits 3 \
#   --sample-s1 20000 \
#   --max-df 50 \
#   --min-len 5 \
#   --neg-per-pos-cap 10 \
#   --test-chunk-size 50000 \
#   --use-faiss \
#   --faiss-top-k 50 \
#   --cache-dir /kaggle/working/cache \
#   --validate

In [ ]:
# Phase 4c — same with Cross-Encoder reranker (AND consensus with LightGBM)
# Requires: !pip install -q transformers torch sentence-transformers
# Best on GPU (T4/A10G). On CPU it's slow; use smaller sample or skip.
# !python $SRC/pipeline.py \
#   --data-dir $DATA_DIR \
#   --out-dir $OUT_ce \
#   --n-splits 3 \
#   --sample-s1 10000 \
#   --max-df 50 \
#   --min-len 5 \
#   --neg-per-pos-cap 10 \
#   --test-chunk-size 20000 \
#   --use-cross-encoder \
#   --cross-encoder-model xlm-roberta-base \
#   --cross-encoder-threshold 0.92 \
#   --consensus-mode and \
#   --cache-dir /kaggle/working/cache \
#   --save-model-dir /kaggle/working/models_ce \
#   --validate

In [ ]:
# Phase 5 — validate + inspect outputs (always before submitting)
!ls -lh $OUT
!head -5 $OUT/matching_results.tsv
!python /kaggle/working/code_ber/business_entity_resolution/utils/validate_submission.py \
  --matching $OUT/matching_results.tsv \
  --candidate $OUT/candidate_pairs.tsv \
  --test-dir $DATA_DIR/test

In [ ]:
# Phase 6 — Full run (after 20k/50k passes). Load saved model, run full test in chunks.
# !python $SRC/pipeline.py \
#   --data-dir $DATA_DIR \
#   --out-dir $OUT_full \
#   --load-model-dir /kaggle/working/models_20k \
#   --test-chunk-size 50000 \
#   --use-faiss \
#   --faiss-top-k 50 \
#   --use-cross-encoder \
#   --cross-encoder-model xlm-roberta-base \
#   --cross-encoder-threshold 0.92 \
#   --consensus-mode and \
#   --validate

In [ ]:
# Optional: Hard-negative mining loop (run after Phase 4 to push F0.5)
# 1. Run pipeline with --save-model-dir
# 2. Extract false positives from validation predictions
# 3. Fine-tune cross-encoder on hard negatives
# 4. Re-run inference with fine-tuned CE
# See: train_ce_hard_negatives.py (to be created)